In [37]:
import pandas as pd
import numpy as np
import pickle

import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
claims_clean = pd.read_pickle("../data/claims-clean.pkl", compression='gzip')
claims_clean.head()

,neo_search_transaction_id,neo_search_subject_id,original_url,text_tmp,internal_feedback,.id,label,mclass,bclass,text_clean
0,12395162.0,11497914.0,http://hosting-22647.tributes.com/obituary/sho...,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 S...",Possible Fatality,url2183,Possible Fatality,Possible Fatality,Relevant claim content,national obituary search click on the item you...
1,12394582.0,11499214.0,https://www.localcrimenews.com/welcome/arrest/...,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",Potentially unlawful activity,url2136,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the following official arrest record for jonat...
2,12384260.0,11485861.0,https://www.bustedmugshots.com/florida/miami-d...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,N/A: No relevant content.,url1586,N/A: No relevant content.,N/A: No relevant content.,N/A: No relevant content.,did someone you know get arrested in miami dad...
3,12363093.0,11481705.0,https://www.policearrests.com/florida-arrest-r...,"<!DOCTYPE HTML>\n<html lang=""en"">\n<head>\n\t<...",Potentially unlawful activity,url209,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the information on this website is taken from ...
4,12376392.0,11488536.0,https://www.bustedmugshots.com/tennessee/memph...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,Potentially unlawful activity,url628,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,name clayton thomas location memphis tennessee...


In [9]:
X_train, X_test, y_train, y_test = train_test_split(claims_clean['text_clean'], claims_clean['bclass'], test_size=0.2, random_state=110122)

#### first tokenization

In [39]:
unigram_vectorizer = TfidfVectorizer(ngram_range=(1,1))

X_unigram_train = unigram_vectorizer.fit_transform(X_train)
X_unigram_test = unigram_vectorizer.transform(X_test)

In [40]:
scaler_unigram = MaxAbsScaler()

X_unigram_train_array = X_unigram_train.toarray()
X_unigram_train_scaled = scaler_unigram.fit_transform(X_unigram_train_array)
X_unigram_test_array = X_unigram_test.toarray()
X_unigram_test_scaled = scaler_unigram.transform(X_unigram_test_array)

pca_unigram = PCA(n_components=100)
X_unigram_train_pca = pca_unigram.fit_transform(X_unigram_train_scaled)
X_unigram_test_pca = pca_unigram.transform(X_unigram_test_scaled)

In [41]:
baseline_model = LogisticRegressionCV(Cs=20, cv=10, penalty="l2", solver="lbfgs", scoring="accuracy", max_iter=5000, n_jobs=-1).fit(X_unigram_train_pca, y_train)

In [42]:
log_odds_train = baseline_model.decision_function(X_unigram_train_pca).reshape(-1, 1)
log_odds_test = baseline_model.decision_function(X_unigram_test_pca).reshape(-1, 1)

In [45]:
test_probs_unigram = baseline_model.predict_proba(X_unigram_test_pca)[:, 1]
test_pred_class_unigram = (test_probs_unigram > 0.5).astype(int)

label_mapping = {0: 'N/A: No relevant content.', 1: 'Relevant claim content'}
y_pred_labels = np.array([label_mapping[p] for p in test_pred_class_unigram])

baseline_accuracy = accuracy_score(y_test, y_pred_labels)
print("Baseline unigram PCR accuracy:", baseline_accuracy)

Baseline unigram PCR accuracy: 0.794392523364486


#### second tokenization

In [46]:
bigram_vectorizer = TfidfVectorizer(ngram_range=(2,2))

X_bigram_train = bigram_vectorizer.fit_transform(X_train)
X_bigram_test = bigram_vectorizer.fit_transform(X_test)

In [47]:
X_bigram_train_array = X_bigram_train.toarray()
X_bigram_train_scaled = StandardScaler(with_mean=False).fit_transform(X_bigram_train_array)
X_bigram_test_array = X_bigram_test.toarray()
X_bigram_test_scaled = StandardScaler(with_mean=False).fit_transform(X_bigram_test_array)

pca_bigram = PCA(n_components=50)
X_bigram_train_pca = pca_bigram.fit_transform(X_bigram_train_scaled)
X_bigram_test_pca = pca_bigram.fit_transform(X_bigram_test_scaled)


In [51]:
X_combined_train = np.hstack([log_odds_train, X_bigram_train_pca])
X_combined_test = np.hstack([log_odds_test, X_bigram_test_pca])

bigram_model = LogisticRegressionCV(Cs=20, cv=10, penalty="l2", solver="lbfgs", scoring="accuracy", max_iter=5000, n_jobs=-1).fit(X_combined_train, y_train)
bigram_model.fit(X_combined_train, y_train)

,Cs,20
,fit_intercept,True
,cv,10
,dual,False
,penalty,'l2'
,scoring,'accuracy'
,solver,'lbfgs'
,tol,0.0001
,max_iter,5000
,class_weight,None
,n_jobs,-1


In [52]:
test_probs_bigram = bigram_model.predict_proba(X_combined_test)[:, 1]
test_pred_class_bigram = (test_probs_bigram > 0.5).astype(int)

y_pred_labels_bigram = np.array([label_mapping[p] for p in test_pred_class_bigram])
bigram_accuracy = accuracy_score(y_test, y_pred_labels_bigram)
print("Bigram PCR accuracy:", bigram_accuracy)


Bigram PCR accuracy: 0.7710280373831776
